# Transform Circuits Data
  1. Read bronze `circuits` table
   1. Keep only the columns required for analytics (Drop `url` column)
   1. Standardise column names using snake_case (`circuitId` → `circuit_id`, `circuitName` → `circuit_name`)
   1. Rename columns to make them more meaningful (`lat` → `latitude`, `long` → `longitude`)
   1. Filter out rows where `circuit_id` is null (business key validation)
   1. Remove duplicate records
   1. Transform values of columns `circuit_name` and `locality` to Title Case
   1. Write the transformed data to silver `circuits` table
 


- ## Step 1 - Read Bronze Table Data

In [0]:
%run ../00.common/01.environment_config

In [0]:
%run ../00.common/02.bronze_helpers

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.circuits"
silver_table = f"{catalog_name}.{silver_schema}.circuits"

In [0]:
circuits_df = spark.table(bronze_table)

## Step 2 : Keep only columns required for Analytics(Drop URL)

In [0]:
circuits_selected_df = circuits_df.drop("url")

## Step 3 &4 - Standardize column name


In [0]:
circuits_renamed_df = (
    circuits_selected_df
    .withColumnsRenamed(
        {"circuitId":"circuit_id",
         "circuitName":"circuit_name",
         "lat":"latitude",
         "long":"longitude"
         
         }
    )
)

## Step 5 - Filter out rows where primary key is NULL

In [0]:
circuits_valid_df = (
    circuits_renamed_df
    .filter(
        F.col("circuit_id").isNotNull()
    )
)


In [0]:
# display(circuits_valid_df)

## Step 6 - Remove Duplicates

In [0]:
circuits_distinct_df = circuits_valid_df.dropDuplicates(["circuit_id"])

In [0]:
# display(circuits_distinct_df)

## Step 7 - Transform required column values to titlecase

In [0]:
circuits_final_df = (
    circuits_distinct_df
    .withColumn("circuit_name", F.initcap(F.col("circuit_name")))
    .withColumn("locality", F.initcap(F.col("locality")))
    
)

In [0]:
# display(circuits_final_df)

## Step 8 - Write data to silver table

In [0]:
(
    circuits_final_df
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(silver_table)
)

In [0]:
# display(spark.table(silver_table))